# QueryToHint Reviewer Figures Notebook


## 1. Setup

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 140
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['axes.grid'] = True

ROOT = Path('.')
OUT = ROOT / 'othertests' / 'paper_figures'
OUT.mkdir(parents=True, exist_ok=True)

DATA_ROOT = ROOT / 'othertests' / 'llmsteer_eval' / 'reviewer_refresh'
BEST_CSV = DATA_ROOT / 'comparison_best_by_model_workload.csv'
ALL_CSV = DATA_ROOT / 'comparison_all_rows.csv'
TABLE5_CSV = DATA_ROOT / 'table5_model_comparison.csv'
HEAVY_SUMMARY = DATA_ROOT / 'heavy_hitters' / 'reptile_vs_contrastive_mpnet_all_summary.json'
HEAVY_TOP20 = DATA_ROOT / 'heavy_hitters' / 'reptile_vs_contrastive_mpnet_all_top20.csv'

required = [BEST_CSV, ALL_CSV, TABLE5_CSV, HEAVY_SUMMARY, HEAVY_TOP20]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        'Missing reviewer artifacts. Run Reviewer_LLMSteer_Pipeline.ipynb first:\n' + '\n'.join(missing)
    )

best = pd.read_csv(BEST_CSV)
all_rows = pd.read_csv(ALL_CSV)
table5 = pd.read_csv(TABLE5_CSV)

MODEL_SHORT = {
    'all_models/finetuned_mpnet_both': 'FT MPNet',
    'reptile_from_finetuned_all/reptile_finetuned_mpnet_both': 'Reptile MPNet',
    'all_models/finetuned_minilm_l12_both': 'FT MiniLM-L12',
    'all_models/finetuned_minilm_l6_both_v2': 'FT MiniLM-L6',
    'all_models/finetuned_gte_small_both': 'FT GTE-small',
    'all_models/finetuned_bge_small_both': 'FT BGE-small',
    'all_models/finetuned_multiqa_minilm_l6_both': 'FT MultiQA',
    'all_models/finetuned_paraphrase_minilm_l6_both': 'FT Paraphrase',
    'stage1_added_models/finetuned_bert_base_both_v2': 'FT BERT',
    'stage1_added_models/finetuned_bge_large_both': 'FT BGE-large',
    'stage1_added_models/finetuned_bge_m3_both': 'FT BGE-M3',
    'models_finetuned/new/distilbert_sql_contrastive': 'FT DistilBERT',
}

def short_label(path: str) -> str:
    return MODEL_SHORT.get(path, Path(path).name)

best['model_short'] = best['model_path'].map(short_label)
all_rows['model_short'] = all_rows['model_path'].map(short_label)
print('best rows:', len(best), '| all rows:', len(all_rows), '| table5 rows:', len(table5))
best.head(3)


## 2. Figure: Runtime Sum by Model (All workload)


In [ ]:
plot_df = best[['model_short','test_workload_sum_mean','test_workload_sum_std']].copy()
plot_df = plot_df.sort_values('test_workload_sum_mean')

plt.figure(figsize=(12,4), constrained_layout=True)
plt.bar(
    plot_df['model_short'],
    plot_df['test_workload_sum_mean'],
    yerr=plot_df['test_workload_sum_std'],
    capsize=4,
    color='#3b82f6'
)
plt.title('All-workload runtime sum by model (lower is better)')
plt.ylabel('Runtime sum')
plt.xticks(rotation=30, ha='right')

fig_path = OUT / 'fig_runtime_sum_all_models.png'
plt.savefig(fig_path, bbox_inches='tight')
plt.show()
print('saved:', fig_path)


## 3. Figure: P90 and Median Runtime by Model


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12,4), constrained_layout=True)

plot_df = best[['model_short','test_p90_mean','test_median_mean']].copy().sort_values('test_p90_mean')

axs[0].bar(plot_df['model_short'], plot_df['test_p90_mean'], color='#6366f1')
axs[0].set_title('P90 runtime (lower is better)')
axs[0].tick_params(axis='x', rotation=30)

plot_df2 = best[['model_short','test_median_mean']].copy().sort_values('test_median_mean')
axs[1].bar(plot_df2['model_short'], plot_df2['test_median_mean'], color='#ef4444')
axs[1].set_title('Median runtime (lower is better)')
axs[1].tick_params(axis='x', rotation=30)

fig_path = OUT / 'fig_p90_median_all_models.png'
plt.savefig(fig_path, bbox_inches='tight')
plt.show()
print('saved:', fig_path)


## 4. Figure: Classification Metrics Trade-off (F1 vs AUROC)


In [ ]:
trade = best[['model_short','f1_mean','auroc_mean']].copy()

plt.figure(figsize=(8,5), constrained_layout=True)
plt.scatter(trade['f1_mean'], trade['auroc_mean'], s=80, color='#0ea5e9')
for _,r in trade.iterrows():
    plt.annotate(r['model_short'], (r['f1_mean'], r['auroc_mean']), fontsize=8, xytext=(5,5), textcoords='offset points')
plt.title('All-workload F1 vs AUROC')
plt.xlabel('F1')
plt.ylabel('AUROC')

fig_path = OUT / 'fig_f1_auroc_tradeoff.png'
plt.savefig(fig_path, bbox_inches='tight')
plt.show()
print('saved:', fig_path)


## 5. Figure: Heavy-Hitter Concentration (Reptile MPNet vs Contrastive MPNet)


In [ ]:
top20 = pd.read_csv(HEAVY_TOP20)
plt.figure(figsize=(8,4))
plt.plot(top20['rank'], top20['cum_abs_mass_fraction'], marker='o', color='#dc2626')
plt.title('Heavy-hitter concentration: Reptile MPNet vs Contrastive MPNet')
plt.xlabel('Top-k queries by |mean delta|')
plt.ylabel('Cumulative abs delta mass fraction')
plt.ylim(0,1.0)

fig_path = OUT / 'fig_heavy_hitter_concentration.png'
plt.savefig(fig_path, bbox_inches='tight')
plt.show()
print('saved:', fig_path)


## 6. Table: Heavy-Hitter Top-20 (for Appendix)


In [ ]:
top20 = pd.read_csv(HEAVY_TOP20)
appendix_cols = [
    'rank','filename','mean_runtime_a','mean_runtime_b',
    'mean_delta','abs_mean_delta','cum_abs_mass_fraction'
]
appendix = top20[appendix_cols].copy()
appendix_path = OUT / 'table_heavy_hitter_top20.csv'
appendix.to_csv(appendix_path, index=False)
print('saved:', appendix_path)
appendix.head(20)


## 7. Export Compact Main Tables


In [ ]:
main_cols = [
    'model_short','config_name',
    'test_workload_sum_mean','test_workload_sum_std',
    'test_p90_mean','test_p90_std',
    'test_median_mean','test_median_std',
    'f1_mean','auroc_mean'
]
main_table = best[main_cols].copy().sort_values('test_workload_sum_mean')
main_path = OUT / 'table_main_results.csv'
main_table.to_csv(main_path, index=False)

# Save the comprehensive all-model reviewer table as well.
table5_path = OUT / 'table5_model_comparison.csv'
table5.to_csv(table5_path, index=False)

print('saved:', main_path)
print('saved:', table5_path)
main_table


## 8. Quick Narrative Snippets (Auto)


In [ ]:
hh = json.loads(HEAVY_SUMMARY.read_text())
best_runtime = best.sort_values('test_workload_sum_mean').iloc[0]
best_f1 = best.sort_values('f1_mean', ascending=False).iloc[0]
best_acc = table5.sort_values('accuracy_mean', ascending=False).iloc[0]

print('Best runtime model:', best_runtime['model_short'], '| sum=', round(best_runtime['test_workload_sum_mean'], 3))
print('Best F1 config model:', best_f1['model_short'], '| F1=', round(best_f1['f1_mean'], 4))
print('Best accuracy (Table 5):', best_acc['model_label'], '| acc=', round(best_acc['accuracy_mean'], 4))
print('Heavy-hitter top1 abs mass fraction:', round(hh['top1_abs_mass_fraction'],4))
print('Heavy-hitter top20 abs mass fraction:', round(hh['top20_abs_mass_fraction'],4))
print('Bootstrap CI mean delta:', hh['bootstrap_ci_mean_delta'])
